# CNN Experiments on the iCoSimal V3 Dataset

This notebook walks through the two mandatory objectives of the Deep Learning MPW CNN project:

**a) Architecture depth** — Train SimpleCNN (2 conv layers), MediumCNN (4 conv layers), and DeepCNN (6 conv layers + BatchNorm) and compare their validation accuracies.

**b) Hyperparameter tuning** — Sweep over image size, learning rate, optimizer, batch size, and dropout to find the best configuration.

## Dataset
The iCoSimal V3 dataset contains 30,000 images of animals in 10 classes:  
`cat, chicken, cow, dog, elephant, horse, rabbit, sheep, squirrel, zebra`  

- 24,000 training images, 6,000 validation images  
- Original resolution: 224×224 px (we resize to 64×64 or 128×128 for speed)

**Download the dataset** from https://drive.switch.ch/index.php/s/NTiYe8mamrgys3M and set `DATA_ROOT` to the path of the directory containing `train/` and `validate/`.

## 0. Setup

In [ ]:
# Install dependencies if running on Colab / a fresh environment
# !pip install -q torch torchvision matplotlib scikit-learn tqdm pandas seaborn

In [ ]:
import sys
import os

# Add the repo root to PYTHONPATH so we can import from src/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
import matplotlib.pyplot as plt

from src.data_loader import get_dataloaders
from src.models import get_model, count_parameters
from src.train import train
from src.evaluate import (
    predict,
    print_classification_report,
    plot_confusion_matrix,
    plot_training_history,
    compare_architectures,
)
from src.hyperparameter_tuning import HparamGrid, run_sweep

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# *** SET THIS PATH TO YOUR DOWNLOADED DATASET ***
DATA_ROOT = "/path/to/icosimal_img_class_03/data_uniform_224_224_sets"

# Quick-experiment settings (change as needed)
IMAGE_SIZE  = 64   # use 128 or 224 for higher accuracy
BATCH_SIZE  = 64
NUM_EPOCHS  = 20   # reduce for faster iteration

## 1. Data Loading

In [ ]:
train_loader, val_loader, CLASS_NAMES = get_dataloaders(
    data_root=DATA_ROOT,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    augment=True,
)
print(f"\nClass names: {CLASS_NAMES}")

In [ ]:
# Visualise a batch of training images
import torchvision
import numpy as np

images, labels = next(iter(train_loader))
# Un-normalise for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
images_show = images[:16].cpu() * std + mean
images_show = images_show.clamp(0, 1)

grid = torchvision.utils.make_grid(images_show, nrow=8, padding=2)
plt.figure(figsize=(14, 4))
plt.imshow(grid.permute(1, 2, 0))
plt.title("Sample training images")
plt.axis("off")
plt.tight_layout()
plt.show()
print("Labels:", [CLASS_NAMES[l] for l in labels[:16].tolist()])

## 2. Architecture Depth Comparison

We train three CNNs of increasing depth with the same hyperparameters and compare their validation accuracy after `NUM_EPOCHS` epochs.

In [ ]:
arch_results = {}

for arch in ("simple", "medium", "deep"):
    print(f"\n{'='*60}")
    print(f" Training {arch.upper()} CNN")
    print(f"{'='*60}")

    model = get_model(
        architecture=arch,
        num_classes=len(CLASS_NAMES),
        input_size=IMAGE_SIZE,
    )
    print(f"Trainable parameters: {count_parameters(model):,}")

    history = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=NUM_EPOCHS,
        learning_rate=1e-3,
        optimizer_name="adam",
        scheduler_name="cosine",
        device=DEVICE,
        verbose=True,
    )
    arch_results[arch] = history
    arch_results[f"{arch}_model"] = model

In [ ]:
# Training history plots per architecture
for arch in ("simple", "medium", "deep"):
    fig = plot_training_history(
        arch_results[arch],
        title=f"{arch.capitalize()} CNN — Training History"
    )
    plt.show()

In [ ]:
# Overlay validation accuracy across all architectures
history_only = {k: v for k, v in arch_results.items() if not k.endswith("_model")}
fig = compare_architectures(history_only, metric="val_acc")
plt.show()

fig = compare_architectures(history_only, metric="val_loss")
plt.show()

In [ ]:
# Summary table
import pandas as pd

rows = []
for arch in ("simple", "medium", "deep"):
    h = arch_results[arch]
    m = arch_results[f"{arch}_model"]
    rows.append({
        "Architecture": arch,
        "Params": f"{count_parameters(m):,}",
        "Best Val Acc (%)": f"{max(h['val_acc']):.2f}",
        "Final Train Acc (%)": f"{h['train_acc'][-1]:.2f}",
    })

pd.DataFrame(rows)

In [ ]:
# Confusion matrix for the deep CNN
deep_model = arch_results["deep_model"]
y_true, y_pred = predict(deep_model, val_loader, DEVICE)
print_classification_report(y_true, y_pred, CLASS_NAMES)

fig = plot_confusion_matrix(y_true, y_pred, CLASS_NAMES,
                            title="Confusion Matrix — Deep CNN")
plt.show()

## 3. Hyperparameter Tuning

We sweep over a grid of hyperparameters.  
Tip: start with `image_size=64` and small `num_epochs` for fast exploration.

In [ ]:
grid = HparamGrid(
    image_size=[64, 128],
    batch_size=[32, 64],
    learning_rate=[1e-3, 3e-4, 1e-4],
    optimizer_name=["adam", "sgd"],
    weight_decay=[1e-4],
    dropout=[0.3, 0.5],
    scheduler_name=["cosine"],
    architecture=["simple", "medium", "deep"],
    num_epochs=[10],   # increase for a more thorough sweep
)

sweep_df = run_sweep(
    data_root=DATA_ROOT,
    grid=grid,
    device=DEVICE,
    verbose=False,
)

sweep_df.to_csv("sweep_results.csv", index=False)
sweep_df.head(15)

In [ ]:
# Visualise the effect of individual hyperparameters on val accuracy
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Hyperparameter Effects on Validation Accuracy", fontsize=14)

hparam_cols = ["architecture", "image_size", "learning_rate",
               "batch_size", "dropout", "optimizer_name"]

for ax, col in zip(axes.flat, hparam_cols):
    sns.boxplot(data=sweep_df, x=col, y="best_val_acc", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Best Val Acc (%)")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 4. Best Model — Final Evaluation

Retrain the best configuration found in the sweep with more epochs and full-resolution (224×224) images.

In [ ]:
best_cfg = sweep_df.iloc[0].to_dict()
print("Best configuration:")
for k, v in best_cfg.items():
    print(f"  {k}: {v}")

In [ ]:
FINAL_IMAGE_SIZE = 128  # or 224 for full resolution
FINAL_EPOCHS     = 30

final_train_loader, final_val_loader, _ = get_dataloaders(
    data_root=DATA_ROOT,
    image_size=FINAL_IMAGE_SIZE,
    batch_size=int(best_cfg["batch_size"]),
    augment=True,
)

best_model = get_model(
    architecture=best_cfg["architecture"],
    num_classes=len(CLASS_NAMES),
    input_size=FINAL_IMAGE_SIZE,
    dropout=float(best_cfg["dropout"]),
)
print(f"Parameters: {count_parameters(best_model):,}")

best_history = train(
    model=best_model,
    train_loader=final_train_loader,
    val_loader=final_val_loader,
    num_epochs=FINAL_EPOCHS,
    learning_rate=float(best_cfg["learning_rate"]),
    optimizer_name=best_cfg["optimizer_name"],
    weight_decay=float(best_cfg["weight_decay"]),
    scheduler_name=best_cfg["scheduler_name"],
    device=DEVICE,
    verbose=True,
)

In [ ]:
fig = plot_training_history(best_history, title="Best Model — Training History")
plt.show()

y_true, y_pred = predict(best_model, final_val_loader, DEVICE)
print_classification_report(y_true, y_pred, CLASS_NAMES)

fig = plot_confusion_matrix(y_true, y_pred, CLASS_NAMES, title="Best Model — Confusion Matrix")
plt.show()

In [ ]:
# Save the trained model
torch.save(best_model.state_dict(), "best_model.pt")
print("Model saved to best_model.pt")